# RL Graveyard — Phase 1 Walkthrough

End-to-end demo: train an agent, autopsy it, look at the death certificate.

Mix of `!` shell cells and Python cells. Run cells top to bottom.

## 1. Environment check

We need Python 3.10+ and the editable-install of this repo.

In [ ]:
!python --version
!pip list 2>/dev/null | grep -E '^(torch|gymnasium|numpy|pytest)' || echo 'Dependencies not yet installed.'

If you haven't installed the package yet (or this notebook is running in a fresh kernel that can't import the project), uncomment and run the next cell once.

In [ ]:
# %cd ..
# !pip install -e '.[dev]'

## 2. Run the test suite

Smoke check that everything imports and the 84 tests pass.

In [ ]:
!cd .. && pytest -q --no-header 2>&1 | tail -5

## 3. Train one agent end-to-end

PPO on CartPole-v1 for 20K steps. Takes ~30s on CPU.

In [ ]:
import os, sys
# Notebook lives in notebooks/; project root is one level up.
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())

In [ ]:
import time
from data.db import init_db
from experiments.config import ExperimentConfig
from experiments.runner import train_and_record

conn = init_db('data/graveyard.sqlite')
cfg = ExperimentConfig(algo='PPO', env='CartPole-v1', seed=0, total_steps=20_000)

t0 = time.time()
run_id = train_and_record(cfg, conn)
print(f'Run {run_id} complete in {time.time() - t0:.1f}s')

## 4. Inspect the autopsy

Use raw SQL via the `sqlite3` CLI to peek at the database.

In [ ]:
!sqlite3 data/graveyard.sqlite -header -column "SELECT r.id, r.algo, r.env, r.seed, a.failure_mode, substr(a.cause, 1, 60) AS cause FROM runs r JOIN autopsies a ON r.id = a.run_id ORDER BY r.id DESC LIMIT 5;"

In [ ]:
import sqlite3, json

conn = sqlite3.connect('data/graveyard.sqlite')
conn.row_factory = sqlite3.Row
row = conn.execute("""
    SELECT r.algo, r.env, r.seed, r.hparams_json, a.failure_mode, a.cause
    FROM runs r JOIN autopsies a ON r.id = a.run_id
    WHERE r.id = ?
""", (run_id,)).fetchone()

print(f"\n{'='*60}")
print(f"  DEATH CERTIFICATE — run {run_id}")
print(f"{'='*60}")
print(f"  Agent:    {row['algo']}")
print(f"  Env:      {row['env']}  (seed {row['seed']})")
print(f"  Verdict:  {row['failure_mode']}")
print(f"  Cause:    {row['cause']}")
print(f"  Hparams:  {json.dumps(json.loads(row['hparams_json']), indent=12)[12:]}")
print(f"{'='*60}")

## 5. Plot the trajectory

Episode returns + entropy over training. Run `pip install matplotlib` if you don't have it.

In [ ]:
import matplotlib.pyplot as plt

eps = conn.execute(
    'SELECT episode_idx, return_ FROM episodes WHERE run_id = ? ORDER BY episode_idx', (run_id,)
).fetchall()
steps = conn.execute(
    'SELECT step, entropy, grad_norm FROM step_metrics WHERE run_id = ? ORDER BY step', (run_id,)
).fetchall()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot([e['episode_idx'] for e in eps], [e['return_'] for e in eps], lw=1)
axes[0].set_title(f"Episode returns — {row['algo']} on {row['env']}")
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Return')

axes[1].plot([s['step'] for s in steps], [s['entropy'] for s in steps], lw=1, color='tab:orange')
axes[1].set_title('Policy entropy')
axes[1].set_xlabel('Env step'); axes[1].set_ylabel('Entropy')

axes[2].plot([s['step'] for s in steps], [s['grad_norm'] for s in steps], lw=1, color='tab:red')
axes[2].set_title('Gradient norm')
axes[2].set_xlabel('Env step'); axes[2].set_ylabel('Grad norm')

plt.tight_layout()
plt.show()

## 6. Mini-sweep

All 6 algorithms × 3 envs × 2 seeds = 36 runs at 20K steps each (~5-10 min total on a single core).

Skip this cell if you just want a single training run.

In [ ]:
from itertools import product
import time

ALGOS = ['PPO', 'A2C', 'REINFORCE', 'DQN', 'DoubleDQN', 'DuelingDQN']
ENVS  = ['CartPole-v1', 'Acrobot-v1', 'RewardTrap-v0']
SEEDS = [0, 1]

total = len(ALGOS) * len(ENVS) * len(SEEDS)
print(f'Running {total} configs...')

for i, (algo, env, seed) in enumerate(product(ALGOS, ENVS, SEEDS), start=1):
    cfg = ExperimentConfig(algo=algo, env=env, seed=seed, total_steps=20_000)
    t0 = time.time()
    try:
        rid = train_and_record(cfg, conn)
        verdict = conn.execute(
            'SELECT failure_mode FROM autopsies WHERE run_id = ?', (rid,)
        ).fetchone()['failure_mode']
        print(f'  [{i:2d}/{total}] {algo:12s} {env:18s} seed={seed}  →  {verdict:22s}  ({time.time()-t0:.1f}s)')
    except sqlite3.IntegrityError:
        print(f'  [{i:2d}/{total}] {algo:12s} {env:18s} seed={seed}  →  already in DB, skipped')

## 7. Verdict distribution

What did the graveyard catch?

In [ ]:
!sqlite3 data/graveyard.sqlite -header -column "SELECT failure_mode, COUNT(*) AS n FROM autopsies GROUP BY failure_mode ORDER BY n DESC;"

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
    SELECT r.algo, r.env, a.failure_mode
    FROM runs r JOIN autopsies a ON r.id = a.run_id
""", conn)

# Crosstab: rows = algo, cols = failure mode
ct = pd.crosstab(df['algo'], df['failure_mode'])
print(ct)

## 8. (Optional) Clear out a run

Useful when iterating. Deletes the run and its dependent rows.

In [ ]:
# DOOMED = run_id  # replace with the run you want to delete
# conn.execute('DELETE FROM step_metrics WHERE run_id = ?', (DOOMED,))
# conn.execute('DELETE FROM episodes WHERE run_id = ?', (DOOMED,))
# conn.execute('DELETE FROM autopsies WHERE run_id = ?', (DOOMED,))
# conn.execute('DELETE FROM runs WHERE id = ?', (DOOMED,))
# conn.commit()

## Next steps

- For the full 200K-step × 5-seed × 6-algo × 5-env grid, use the Slurm array job pattern described in the previous chat or wait for Phase 2's runner.
- Phase 2 will add `experiments/grid.yaml` + a proper sweep entrypoint.
- Phase 3 reads from this same `data/graveyard.sqlite` to populate the interactive frontend.